# scEEMS Prediction

Apply a trained feature-weighted CatBoost scEEMS model to score genetic variants for their probability of having a functional, cell-type-specific regulatory impact on gene expression. This worker runs the `predict` subcommand of `gems_pipeline.py` and emits per-variant Expression Modifier Scores.

**Author:** Angelina Siagailo, with input from Gao Wang and Katie Cho. Method and model are described in the scEEMS manuscript; the upstream training step is documented in [EMS Training](https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_training.html).

## Description

The trained CatBoost classifier assigns each variant a continuous EMS score in `[0, 1]` representing the probability that it modifies gene expression in the target cell type. Suggested prioritization tiers: score > 0.8 high (prioritize for experimental validation such as CRISPR screens or luciferase assays); 0.5-0.8 moderate (context-dependent); < 0.5 low. New cohorts / cell types are supported by editing the YAML configs only - no Python code changes are required (e.g. swap `protocol_example` for `Ast_mega_eQTL`).

**Timing:** ~varies on typical compute infrastructure.

## Input

| File | Description |
|------|-------------|
| `--model_path` | Trained feature-weighted CatBoost model (`.joblib`) produced by the [EMS Training](https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_training.html) workflow for the chosen cohort / chromosome. |
| `--data_config` (`data_config.yaml`) | Data configuration: cohort, paths to the variant list to score, and the full 4,839-feature set. New cohorts/cell types are added by editing this YAML only. |
| `cohort` (positional) | Cohort / cell type, e.g. `protocol_example`. Must match an entry in the data config. |
| `chromosome` (positional) | Chromosome to score, e.g. `2`. |

The variant list referenced by the data config is a tab-separated file of `variant_id` entries in `chr:pos:ref:alt` notation (1-based GRCh38 coordinates, ACGT alleles), e.g. `2:12345:A:T`.

## Output

| File | Description |
|------|-------------|
| `predictions_weighted_model_chr<chromosome>.tsv` | Per-variant predictions: `variant_id`, the continuous EMS score column (`standard_subset_weighted_pred_prob`), and a binary label column. |

The continuous score column is the primary result used to rank and prioritize variants for downstream experimental validation.

## Steps

**Step 1.** Score variants with a trained feature-weighted CatBoost scEEMS model for one cohort / chromosome. The toy command below scores the microglia cohort (`protocol_example`) on chromosome 2 using a model produced by the training workflow.

In [ ]:
sos run pipeline/ems_prediction.ipynb predict \
    --cohort protocol_example \
    --chromosome 2 \
    --model_path output/ems_training/protocol_example_chr2_scEEMS_model.joblib \
    --data_config code/xqtl_modifier_score/data_config.yaml \
    --cwd output/ems_prediction


## Command interface

In [ ]:
sos run pipeline/ems_prediction.ipynb -h

## Workflow implementation

The `predict` step wraps the `gems_pipeline.py predict` engine. The pipeline loads the trained model, annotates the input variants with the full feature set, applies the 10x deep-learning feature weighting used at training time, and writes per-variant EMS scores. New datasets / cell types are scored by editing the `data_config` YAML only - no Python code changes are required.

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[global]
import os
# Work directory & output directory
parameter: cwd = path('output/ems_prediction')
# Cohort / cell type to score (must match an entry in the data config)
parameter: cohort = 'protocol_example'
# Chromosome to score
parameter: chromosome = '2'
# Trained CatBoost model (.joblib) from the EMS Training workflow
parameter: model_path = path('output/ems_training/protocol_example_chr2_scEEMS_model.joblib')
# Data configuration YAML (cohort, variant list, feature set)
parameter: data_config = path('code/xqtl_modifier_score/data_config.yaml')
# Directory holding gems_pipeline.py
parameter: pipeline_dir = path('code/xqtl_modifier_score')
parameter: job_size = 1
parameter: mem = '60G'
parameter: walltime = '24h'


In [ ]:
[predict]
output: f'{pipeline_dir:a}/' + __import__('yaml').safe_load(open(data_config))['output']['base_dir'].format(cohort=cohort) + f"/{__import__('yaml').safe_load(open(data_config))['output']['predictions_dir']}/predictions_weighted_model_chr{chromosome}.tsv"
task: trunk_workers = 1, trunk_size = job_size, mem = mem, walltime = walltime, tags = f'{step_name}_{cohort}_chr{chromosome}'
bash: expand = "$[ ]", workdir = pipeline_dir, stderr = f'{_output[0]}.stderr', stdout = f'{_output[0]}.stdout'
    python gems_pipeline.py predict $[cohort] $[chromosome] \
        --model_path $[model_path:a] \
        --data_config $[data_config:a]